## Resources — jobs, pipelines & other bundle-managed objects

Each resource type has a schema that **mirrors the corresponding Databricks REST API**. A job in a bundle uses the same fields you'd send to `POST /api/2.1/jobs/create` (module 06) — so the YAML you learned to *read* there is the YAML you *author* here.

`resources/job_nightly.yml`:

```yaml
resources:
  jobs:
    fintech_nightly_ingest:
      schedule: { quartz_cron_expression: "0 0 2 * * ?", timezone_id: "UTC" }
      email_notifications: { on_failure: ["oncall@bank.example"] }
      tasks:
        - task_key: ingest_cards
          notebook_task:
            notebook_path: ../notebooks/ingest_cards.ipynb
            base_parameters: { catalog: ${var.catalog} }
          job_cluster_key: shared_etl_cluster
          max_retries: 3
      job_clusters:
        - job_cluster_key: shared_etl_cluster
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            node_type_id: ${var.node_type_id}
            num_workers: 4
```

`resources/pipeline_card_etl.yml` — a declarative pipeline:

```yaml
resources:
  pipelines:
    card_etl:
      name: card_etl_${bundle.target}
      target: ${var.catalog}.gold
      libraries: [{ notebook: { path: ../src/pipeline.py } }]
```

Notice `${var.catalog}` and `${bundle.target}` — values flow from the manifest into every resource. **What a bundle can define:** jobs, pipelines, ML experiments & models, dashboards, volumes, and schemas — each keyed under `resources:` by type. Splitting them into `resources/*.yml` files (pulled in via `include`) keeps the manifest readable as the project grows.
